In [1]:
import re

#Creating a tokenizer

def tokenize(text):
    text=text.lower()

    tokens=re.findall(r"\b\w+\b|[^\W\s]",text)

    return tokens

In [2]:
sentence="I love Artificial intelligence"

tokens = tokenize(sentence)

print("Original Text:",sentence)
print("Tokens:",tokens)
print("Number of Tokens:",len(tokens))

Original Text: I love Artificial intelligence
Tokens: ['i', 'love', 'artificial', 'intelligence']
Number of Tokens: 4


In [5]:
sentences = [
    "Hello World!",
    "Hello,How are you?",
    "Artificial Intelligence is fascinating.",
    "I am learning Python for AI engineering.",
    "Tokenization converts text into tokens."
]

for sentence in sentences:
    tokens = tokenize(sentence)

    print("\nText:", sentence)
    print("Tokens:", tokens)
    print("Count:", len(tokens))


Text: Hello World!
Tokens: ['hello', 'world']
Count: 2

Text: Hello,How are you?
Tokens: ['hello', 'how', 'are', 'you']
Count: 4

Text: Artificial Intelligence is fascinating.
Tokens: ['artificial', 'intelligence', 'is', 'fascinating']
Count: 4

Text: I am learning Python for AI engineering.
Tokens: ['i', 'am', 'learning', 'python', 'for', 'ai', 'engineering']
Count: 7

Text: Tokenization converts text into tokens.
Tokens: ['tokenization', 'converts', 'text', 'into', 'tokens']
Count: 5


In [7]:
#Converting token into token IDs

vocabulary={
    "i":1,
    "love":2,
    "learning":3,
    "artificial":4,
    "intelligence":5,
    "!":6
}

sentence="I love learning artificial intelligence!"

tokens=tokenize(sentence)

token_ids=[]

for token in tokens:
    token_id=vocabulary[token]
    token_ids.append(token_id)

print("Tokens:",tokens)
print("Token IDs",token_ids)

Tokens: ['i', 'love', 'learning', 'artificial', 'intelligence']
Token IDs [1, 2, 3, 4, 5]


In [8]:
#making the above thing as a reusable function
def encode(tokens,vocabulary):
    token_ids=[]

    for token in tokens:
        token_ids.append(vocabulary[token])

    return token_ids

token_ids=encode(tokens,vocabulary)

print(token_ids)

[1, 2, 3, 4, 5]


In [10]:
#What happends when the sentence contains a word that is not in your vocabulary?

sentence="I love Python"

tokens=tokenize(sentence)
token_ids=encode(tokens,vocabulary)

print(token_ids)  #Will get a KeyError

KeyError: 'python'

In [11]:
# A tokenizer must ahve a strategy for handling words that are not in it vocabulary.
#Fixing it using an Unknown Token <UNK>

vocabulary={
    "<UNK>":0,
    "i":1,
    "love":2,
    "learning":3,
    "Artificial":4,
    "Intelligence":5,
    "!":6
}


#updating our encoding fucntion
def encode(tokens,vocabulary):
    token_ids=[]

    for token in tokens:
        token_id=vocabulary.get(token,vocabulary["<UNK>"])

    token_ids.append(token_id)

    return token_ids

In [12]:
#Testing Updated tokenizer
sentence="I love Python"
tokens=tokenize(sentence)
token_ids=encode(tokens,vocabulary)

print("Tokens:",tokens)
print("Token IDs:",token_ids)


Tokens: ['i', 'love', 'python']
Token IDs: [0]


In [13]:
#Experiment

sentences = [
    "I love learning",
    "I love Python",
    "Python is artificial intelligence"
]

for sentence in sentences:
    tokens = tokenize(sentence)
    token_ids = encode(tokens, vocabulary)

    print("\nSentence:", sentence)
    print("Tokens:", tokens)
    print("Token IDs:", token_ids)


Sentence: I love learning
Tokens: ['i', 'love', 'learning']
Token IDs: [3]

Sentence: I love Python
Tokens: ['i', 'love', 'python']
Token IDs: [0]

Sentence: Python is artificial intelligence
Tokens: ['python', 'is', 'artificial', 'intelligence']
Token IDs: [0]


### Building  a Simple subword tokenizer

In [14]:
#Creating Vocabulary
subword_vocabulary = {
    "<UNK>": 0,
    "i": 1,
    "love": 2,
    "play": 3,
    "ing": 4,
    "un": 5,
    "happy": 6,
    "er": 7,
    "ed": 8,
    "ful": 9,
    "py": 10,
    "thon": 11,
    "!": 12,
    ".": 13
}


In [15]:
#Building the sub word matching function

def split_into_subwords(word, vocabulary):
    pieces = []
    position = 0

    # Ignore the special unknown token when matching
    known_pieces = [
        piece for piece in vocabulary
        if piece != "<UNK>"
    ]

    while position < len(word):
        match = None

        # Try the longest vocabulary piece first
        for piece in sorted(known_pieces, key=len, reverse=True):
            if word.startswith(piece, position):
                match = piece
                break

        if match:
            pieces.append(match)
            position += len(match)
        else:
            pieces.append("<UNK>")
            position += 1

    return pieces

In [16]:
#Testing individual words
words = [
    "playing",
    "unhappy",
    "player",
    "played",
    "python"
]

for word in words:
    pieces = split_into_subwords(word, subword_vocabulary)

    print(word, "->", pieces)

playing -> ['play', 'ing']
unhappy -> ['un', 'happy']
player -> ['play', 'er']
played -> ['play', 'ed']
python -> ['py', 'thon']


In [18]:
#Coverting subword into tokens
def encode_subwords(pieces, vocabulary):
    return [
        vocabulary.get(piece, vocabulary["<UNK>"])
        for piece in pieces
    ]


word = "playing"

pieces = split_into_subwords(word, subword_vocabulary)
token_ids = encode_subwords(pieces, subword_vocabulary)

print("Word:", word)
print("Subwords:", pieces)
print("Token IDs:", token_ids)

Word: playing
Subwords: ['play', 'ing']
Token IDs: [3, 4]


In [19]:
#What happens with unfamiliar word

word = "unpredictable"

pieces = split_into_subwords(word, subword_vocabulary)
token_ids = encode_subwords(pieces, subword_vocabulary)

print("Word:", word)
print("Pieces:", pieces)
print("IDs:", token_ids)

Word: unpredictable
Pieces: ['un', '<UNK>', '<UNK>', 'ed', 'i', '<UNK>', '<UNK>', '<UNK>', '<UNK>', '<UNK>', '<UNK>']
IDs: [5, 0, 0, 8, 1, 0, 0, 0, 0, 0, 0]


### Real LLM Tokenizer

In [1]:
!pip install tiktoken

   ---------------------------------------- 0.0/941.2 kB ? eta -:--:--
   ---------------------------------------- 0.0/941.2 kB ? eta -:--:--
   ---------------------------------------- 0.0/941.2 kB ? eta -:--:--
   ----------- ---------------------------- 262.1/941.2 kB ? eta -:--:--
   --------------------- ---------------- 524.3/941.2 kB 762.0 kB/s eta 0:00:01
   --------------------- ---------------- 524.3/941.2 kB 762.0 kB/s eta 0:00:01
   ------------------------------- ------ 786.4/941.2 kB 780.2 kB/s eta 0:00:01
   -------------------------------------- 941.2/941.2 kB 804.6 kB/s eta 0:00:00



[notice] A new release of pip is available: 25.0.1 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import tiktoken

encoding = tiktoken.get_encoding("cl100k_base")

text = "I love learning artificial intelligence!"

tokens = encoding.encode(text)

print("Token IDs:", tokens)
print("Token count:", len(tokens))
print("Decoded text:", encoding.decode(tokens))

Token IDs: [40, 3021, 6975, 21075, 11478, 0]
Token count: 6
Decoded text: I love learning artificial intelligence!


### Implementing a simplified Byte Pair Encoding Algorithm

In [3]:
#Preapring a training corpus
corpus = [
    "low",
    "lower",
    "lowest",
    "low",
    "lower",
    "newer",
    "wider"
]

# Split each word into individual characters
words = [list(word) for word in corpus]

print(words)

[['l', 'o', 'w'], ['l', 'o', 'w', 'e', 'r'], ['l', 'o', 'w', 'e', 's', 't'], ['l', 'o', 'w'], ['l', 'o', 'w', 'e', 'r'], ['n', 'e', 'w', 'e', 'r'], ['w', 'i', 'd', 'e', 'r']]


In [4]:
#Count adjacent pairs

from collections import Counter

def get_pair_counts(words):
    pairs = Counter()

    for word in words:
        for i in range(len(word) - 1):
            pair = (word[i], word[i + 1])
            pairs[pair] += 1

    return pairs


pair_counts = get_pair_counts(words)

print(pair_counts)

Counter({('l', 'o'): 5, ('o', 'w'): 5, ('w', 'e'): 4, ('e', 'r'): 4, ('e', 's'): 1, ('s', 't'): 1, ('n', 'e'): 1, ('e', 'w'): 1, ('w', 'i'): 1, ('i', 'd'): 1, ('d', 'e'): 1})


In [5]:
#Merge a selected pair

def merge_pair(words, pair):
    merged_words = []

    target = list(pair)
    merged_symbol = "".join(pair)

    for word in words:
        new_word = []
        i = 0

        while i < len(word):
            if word[i:i + 2] == target:
                new_word.append(merged_symbol)
                i += 2
            else:
                new_word.append(word[i])
                i += 1

        merged_words.append(new_word)

    return merged_words

In [6]:
# Train BPE through repeated merges

words = [list(word) for word in corpus]
merge_rules = []

for step in range(10):
    pair_counts = get_pair_counts(words)

    if not pair_counts:
        break

    most_common_pair = pair_counts.most_common(1)[0][0]

    words = merge_pair(words, most_common_pair)

    merge_rules.append(most_common_pair)

    print(f"Step {step + 1}:")
    print("Merged pair:", most_common_pair)
    print("Corpus:", words)
    print()

Step 1:
Merged pair: ('l', 'o')
Corpus: [['lo', 'w'], ['lo', 'w', 'e', 'r'], ['lo', 'w', 'e', 's', 't'], ['lo', 'w'], ['lo', 'w', 'e', 'r'], ['n', 'e', 'w', 'e', 'r'], ['w', 'i', 'd', 'e', 'r']]

Step 2:
Merged pair: ('lo', 'w')
Corpus: [['low'], ['low', 'e', 'r'], ['low', 'e', 's', 't'], ['low'], ['low', 'e', 'r'], ['n', 'e', 'w', 'e', 'r'], ['w', 'i', 'd', 'e', 'r']]

Step 3:
Merged pair: ('e', 'r')
Corpus: [['low'], ['low', 'er'], ['low', 'e', 's', 't'], ['low'], ['low', 'er'], ['n', 'e', 'w', 'er'], ['w', 'i', 'd', 'er']]

Step 4:
Merged pair: ('low', 'er')
Corpus: [['low'], ['lower'], ['low', 'e', 's', 't'], ['low'], ['lower'], ['n', 'e', 'w', 'er'], ['w', 'i', 'd', 'er']]

Step 5:
Merged pair: ('low', 'e')
Corpus: [['low'], ['lower'], ['lowe', 's', 't'], ['low'], ['lower'], ['n', 'e', 'w', 'er'], ['w', 'i', 'd', 'er']]

Step 6:
Merged pair: ('lowe', 's')
Corpus: [['low'], ['lower'], ['lowes', 't'], ['low'], ['lower'], ['n', 'e', 'w', 'er'], ['w', 'i', 'd', 'er']]

Step 7:
Merged 

In [7]:
#Inspecting the learned merges
print("Learned BPE merge rules:")

for rule in merge_rules:
    print(rule, "->", "".join(rule))

Learned BPE merge rules:
('l', 'o') -> lo
('lo', 'w') -> low
('e', 'r') -> er
('low', 'er') -> lower
('low', 'e') -> lowe
('lowe', 's') -> lowes
('lowes', 't') -> lowest
('n', 'e') -> ne
('ne', 'w') -> new
('new', 'er') -> newer
